In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Проходи транспілятора на основі ШІ
Проходи транспілятора на основі ШІ — це проходи, які слугують повноцінною заміною «традиційних» проходів Qiskit для деяких задач транспіляції. Вони нерідко дають кращі результати, ніж існуючі евристичні алгоритми (зокрема меншу глибину та кількість CNOT-вентилів), і водночас є значно швидшими за оптимізаційні алгоритми на кшталт розв'язувачів булевої виконуваності. Проходи транспілятора на основі ШІ можна запускати у локальному середовищі або в хмарі через Qiskit Transpiler Service — якщо ти маєш доступ до IBM Quantum&reg; Premium Plan, Flex Plan або On-Prem (через IBM Quantum Platform API) Plan.

> **Note:** Проходи транспілятора на основі ШІ перебувають у статусі бета-версії та можуть змінюватися.
>     Якщо маєш відгуки або хочеш зв'язатися з командою розробників, скористайся [каналом у Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU).

На сьогодні доступні такі проходи:

**Проходи маршрутизації**
 - `AIRouting`: вибір розміщення та маршрутизація схеми

**Проходи синтезу схем**
 - `AICliffordSynthesis`: синтез схем Кліффорда
 - `AILinearFunctionSynthesis`: синтез схем лінійних функцій
 - `AIPermutationSynthesis`: синтез схем перестановок
 - `AIPauliNetworkSynthesis`: синтез схем мережі Паулі

Щоб використовувати проходи транспілятора на основі ШІ, спочатку [встанови пакет `qiskit-ibm-transpiler`](/guides/qiskit-transpiler-service#install-transpiler-service). Більше інформації про доступні параметри можна знайти в [документації API qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler).

## Запуск проходів транспілятора на основі ШІ локально або в хмарі
Якщо хочеш безкоштовно використовувати проходи транспілятора на основі ШІ у локальному середовищі, встанови `qiskit-ibm-transpiler` з додатковими залежностями:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService

backend = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Без цих додаткових залежностей проходи транспілятора на основі ШІ запускаються в хмарі через Qiskit Transpiler Service (доступно лише для користувачів IBM Quantum Premium Plan, Flex Plan або On-Prem (через IBM Quantum Platform API) Plan). Після встановлення додаткових залежностей типовим режимом запуску проходів є локальна машина.

## Прохід маршрутизації ШІ
Прохід `AIRouting` виконує функції як етапу розміщення, так і етапу маршрутизації. Його можна використовувати у `PassManager` таким чином:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit_ibm_transpiler.ai.synthesis import AIPauliNetworkSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectPauliNetworks
from qiskit.circuit.library import efficient_su2

ibm_torino = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_torino,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Linear Function blocks
        CollectPauliNetworks(),  # Collect Pauli Networks blocks
        AIPauliNetworkSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Pauli Network blocks.
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Тут `backend` визначає карту зв'язності для маршрутизації, `optimization_level` (1, 2 або 3) задає обчислювальні зусилля в процесі (вищий рівень зазвичай дає кращий результат, але займає більше часу), а `layout_mode` вказує спосіб вибору розміщення.
`layout_mode` підтримує такі опції:

- `keep`: Зберігає розміщення, задане попередніми проходами транспілятора (або використовує тривіальне розміщення, якщо воно не задане). Зазвичай застосовується лише тоді, коли схему потрібно запустити на конкретних кубітах пристрою. Часто дає гірші результати через менший простір для оптимізації.
- `improve`: Використовує розміщення, задане попередніми проходами транспілятора, як відправну точку. Корисний, коли є хороше початкове здогадування щодо розміщення — наприклад, для схем, побудованих так, що вони приблизно відповідають карті зв'язності пристрою. Також стане в пригоді, якщо хочеш спробувати інші конкретні проходи розміщення в поєднанні з `AIRouting`.
- `optimize`: Типовий режим. Найкраще підходить для загальних схем, де немає хороших здогадок щодо розміщення. Цей режим ігнорує попередні вибори розміщення.
- `local_mode`: Цей прапорець визначає, де виконується прохід `AIRouting`. Якщо `False` — `AIRouting` запускається віддалено через Qiskit Transpiler Service. Якщо `True` — пакет намагається запустити прохід у локальному середовищі з відкатом до хмарного режиму, якщо потрібні залежності не знайдено.

## Проходи синтезу схем ШІ
Проходи синтезу схем ШІ дають змогу оптимізувати фрагменти схем різних типів ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Pauli Network) шляхом їх повторного синтезу. Типовий спосіб використання проходу синтезу виглядає так:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_torino")
torino_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=torino_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

Синтез поважає карту зв'язності пристрою: його можна безпечно запускати після інших проходів маршрутизації, не порушуючи схему, тому загальна схема все одно відповідатиме обмеженням пристрою. За замовчуванням синтез замінює оригінальну підсхему лише тоді, коли синтезована підсхема є кращою (наразі перевіряється лише кількість CNOT), але це можна змінити, щоб заміна відбувалась завжди, задавши `replace_only_if_better=False`.

Наступні проходи синтезу доступні з `qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: синтез для схем [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford) (блоки вентилів `H`, `S` і `CX`). Наразі підтримуються блоки до дев'яти кубітів.
- *AILinearFunctionSynthesis*: синтез для схем [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) (блоки вентилів `CX` і `SWAP`). Наразі підтримуються блоки до дев'яти кубітів.
- *AIPermutationSynthesis*: синтез для схем [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation) (блоки вентилів `SWAP`). Наразі доступний для блоків на 65, 33 і 27 кубітів.
- *AIPauliNetworkSynthesis*: синтез для схем мережі Паулі (блоки вентилів `H`, `S`, `SX`, `CX`, `RX`, `RY` і `RZ`). Наразі підтримуються блоки до шести кубітів.

Ми плануємо поступово збільшувати розмір підтримуваних блоків.

Усі проходи використовують пул потоків для паралельного надсилання кількох запитів. За замовчуванням максимальна кількість потоків дорівнює кількості ядер плюс чотири (типові значення для об'єкта Python `ThreadPoolExecutor`). Однак ти можеш задати власне значення за допомогою аргументу `max_threads` під час ініціалізації проходу. Наприклад, наступний рядок ініціалізує прохід `AILinearFunctionSynthesis`, дозволяючи йому використовувати максимум 20 потоків.